# MNIST 1D — Full CRISP-DM Cycle with PyTorch + MLflow

**Dataset:** [MNIST 1D](https://github.com/greydanus/mnist1d) — 40-point 1D sequences, 10 digit classes, 4 000 train / 1 000 test samples.  
**Model:** 3-layer 1D CNN with BatchNorm and Adaptive Average Pooling.  
**Stack:** JupyterLab → MLflow tracking server → MinIO artifact store → DVC remote cache.

| CRISP-DM Phase | Notebook section |
|---|---|
| Business Understanding | § 1 |
| Data Understanding | § 2 – Explore |
| Data Preparation | § 3 – Prepare |
| Modelling | § 4 – Define model |
| Evaluation | § 5 – Train & Evaluate |
| Deployment | § 6 – Log to MLflow / DVC |


In [ ]:
import subprocess, sys

packages = [
    'mnist1d', 'torch', 'matplotlib', 'seaborn',
    'scikit-learn', 'mlflow', 'boto3',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + packages)
print('All packages ready.')


In [ ]:
import os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.manifold import TSNE
import mlflow
import mlflow.pytorch

warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5000')
EXPERIMENT_NAME     = 'mnist1d-cnn'
RUN_NAME            = 'conv1d-3layers'

SEED         = 42
EPOCHS       = 40
BATCH_SIZE   = 128
LR           = 1e-3
WEIGHT_DECAY = 1e-4

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device            : {DEVICE}')
print(f'MLflow server     : {MLFLOW_TRACKING_URI}')
print(f'Experiment        : {EXPERIMENT_NAME}')


## § 1 Business Understanding

**Goal:** Classify 1D digit signals into 10 classes with high accuracy and full experiment traceability.

MNIST 1D compresses the familiar 28×28 MNIST images into 40-point 1D sequences using a parametric template + noise process. This makes it a fast, low-resource benchmark that still separates weak from strong inductive biases — a linear model gets ~50 %, an MLP ~68 %, and a 1D CNN should exceed 95 %.

**Success criteria:**
- Test accuracy ≥ 94 %
- All metrics, hyperparameters, and artifacts logged in MLflow
- Confusion matrix stored as a PNG artifact in MinIO
- Dataset versioned with DVC


## § 2 Data Understanding

In [ ]:
def load_mnist1d():
    cache = 'mnist1d_data.pkl'
    if os.path.exists(cache):
        with open(cache, 'rb') as f:
            return pickle.load(f)
    try:
        from mnist1d.data import get_dataset, get_dataset_args
        args = get_dataset_args()
        return get_dataset(args, path=cache, download=True, regenerate=False)
    except Exception:
        import urllib.request
        url = 'https://github.com/greydanus/mnist1d/raw/master/mnist1d_data.pkl'
        print(f'Downloading from {url} ...')
        urllib.request.urlretrieve(url, cache)
        with open(cache, 'rb') as f:
            return pickle.load(f)

data = load_mnist1d()

X_train = np.array(data['x'],      dtype=np.float32)   # (4000, 40)
y_train = np.array(data['y'],      dtype=np.int64)
X_test  = np.array(data['x_test'], dtype=np.float32)   # (1000, 40)
y_test  = np.array(data['y_test'], dtype=np.int64)

SEQ_LEN   = X_train.shape[1]
N_CLASSES = len(np.unique(y_train))

print(f'Train  : {X_train.shape}  labels {np.unique(y_train)}')
print(f'Test   : {X_test.shape}')
print(f'Seq len: {SEQ_LEN}  Classes: {N_CLASSES}')
print(f'Signal stats — mean: {X_train.mean():.3f}  std: {X_train.std():.3f}  '
      f'min: {X_train.min():.3f}  max: {X_train.max():.3f}')


In [ ]:
# Class distribution + sample waveforms
unique, counts = np.unique(y_train, return_counts=True)
colors = plt.cm.tab10(np.linspace(0, 1, 10))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].bar(unique, counts, color=colors)
axes[0].set_xlabel('Digit class'); axes[0].set_ylabel('Count')
axes[0].set_title('Class distribution (training set)')
axes[0].set_xticks(range(10))

for cls in range(10):
    idx = np.where(y_train == cls)[0][0]
    axes[1].plot(X_train[idx] + cls * 2, label=str(cls), color=colors[cls], alpha=0.85)
axes[1].set_xlabel('Time step')
axes[1].set_title('One sample per class (offset for clarity)')
axes[1].legend(title='Digit', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

plt.suptitle('MNIST 1D — Data Understanding', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data_exploration.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# 5 samples per class grid
fig, axes = plt.subplots(2, 5, figsize=(16, 5))
for cls, ax in enumerate(axes.flat):
    idxs = np.where(y_train == cls)[0][:5]
    for i in idxs:
        ax.plot(X_train[i], alpha=0.5, color=plt.cm.tab10(cls / 10))
    ax.set_title(f'Digit {cls}', fontweight='bold')
    ax.set_ylim(-0.3, 1.3); ax.set_xticks([])
plt.suptitle('5 samples per digit class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('class_samples.png', dpi=120, bbox_inches='tight')
plt.show()


## § 3 Data Preparation

Steps:
1. Normalise to zero mean / unit std (computed on training set only).
2. Add a channel dimension: `(N, 40)` → `(N, 1, 40)` for `Conv1d`.
3. Wrap in `TensorDataset` + `DataLoader`.


In [ ]:
X_mean, X_std = X_train.mean(), X_train.std()
X_train_n = (X_train - X_mean) / X_std
X_test_n  = (X_test  - X_mean) / X_std

# (N, 40) → (N, 1, 40)
X_tr = torch.from_numpy(X_train_n[:, None, :])
y_tr = torch.from_numpy(y_train)
X_te = torch.from_numpy(X_test_n[:, None, :])
y_te = torch.from_numpy(y_test)

train_loader = DataLoader(TensorDataset(X_tr, y_tr),
                          batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader  = DataLoader(TensorDataset(X_te, y_te),
                          batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}  Test batches: {len(test_loader)}')
print(f'After normalisation — mean: {X_train_n.mean():.4f}  std: {X_train_n.std():.4f}')


## § 4 Modelling

**Architecture:** 3 × Conv1d blocks with BatchNorm, ReLU, and MaxPool, followed by Adaptive Average Pooling to collapse the time axis, then a two-layer MLP head.

```
Input  (B, 1, 40)
  Conv1d(1→32, k=5, p=2) + BN + ReLU + MaxPool1d(2)  → (B, 32, 20)
  Conv1d(32→64, k=3, p=1) + BN + ReLU + MaxPool1d(2)  → (B, 64, 10)
  Conv1d(64→128, k=3, p=1) + BN + ReLU               → (B, 128, 10)
  AdaptiveAvgPool1d(1)                                 → (B, 128, 1)
  Flatten                                              → (B, 128)
  Linear(128→64) + ReLU + Dropout(0.3)
  Linear(64→10)                                        → logits
```


In [ ]:
class Conv1DClassifier(nn.Module):
    def __init__(self, n_classes: int = 10, dropout: float = 0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1,   32, kernel_size=5, padding=2), nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32,  64, kernel_size=3, padding=1), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

    def encode(self, x):
        """128-dim feature vector (before the MLP head)."""
        return self.features(x).squeeze(-1)


model = Conv1DClassifier().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

# Shape sanity check
dummy = torch.randn(4, 1, SEQ_LEN).to(DEVICE)
assert model(dummy).shape == (4, N_CLASSES), 'Unexpected output shape'
assert model.encode(dummy).shape == (4, 128), 'Unexpected encoder shape'
print(f'Output shape: {model(dummy).shape}  ✓')


## § 5 Training & Evaluation

In [ ]:
def train_epoch(model, loader, optimiser, criterion):
    model.train()
    total_loss = correct = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimiser.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimiser.step()
        total_loss += loss.item() * len(yb)
        correct    += (model(xb).argmax(1) == yb).sum().item()
        n          += len(yb)
    return total_loss / n, correct / n


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = correct = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        total_loss += criterion(logits, yb).item() * len(yb)
        correct    += (logits.argmax(1) == yb).sum().item()
        n          += len(yb)
    return total_loss / n, correct / n


In [ ]:
criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

with mlflow.start_run(run_name=RUN_NAME) as run:
    RUN_ID = run.info.run_id
    mlflow.log_params({
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR,
        'weight_decay': WEIGHT_DECAY, 'optimizer': 'Adam+CosineAnnealing',
        'architecture': 'Conv1D-3layers', 'n_params': n_params,
        'seed': SEED, 'seq_len': SEQ_LEN, 'n_classes': N_CLASSES,
    })

    print(f'Run ID: {RUN_ID}')
    print(f'{"Epoch":>5}  {"Train Loss":>10}  {"Train Acc":>9}  '
          f'{"Val Loss":>8}  {"Val Acc":>7}')
    print('-' * 55)

    best_val_acc = 0.0
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimiser, criterion)
        va_loss, va_acc = eval_epoch(model, test_loader, criterion)
        scheduler.step()

        for k, v in zip(
            ['train_loss', 'train_acc', 'val_loss', 'val_acc'],
            [tr_loss, tr_acc, va_loss, va_acc],
        ):
            history[k].append(v)

        mlflow.log_metrics({
            'train_loss': tr_loss, 'train_acc': tr_acc,
            'val_loss': va_loss,   'val_acc': va_acc,
            'lr': scheduler.get_last_lr()[0],
        }, step=epoch)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), 'best_model.pt')

        if epoch % 5 == 0 or epoch == 1:
            print(f'{epoch:>5}  {tr_loss:>10.4f}  {tr_acc:>9.3%}  '
                  f'{va_loss:>8.4f}  {va_acc:>7.3%}')

    mlflow.log_metric('best_val_acc', best_val_acc)

print(f'\nBest validation accuracy: {best_val_acc:.3%}')
print(f'MLflow UI: {MLFLOW_TRACKING_URI}')


In [ ]:
# Training curves
epochs = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, history['train_loss'], label='Train', color='#2196F3')
ax1.plot(epochs, history['val_loss'],   label='Val',   color='#F44336')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, [a * 100 for a in history['train_acc']], label='Train', color='#2196F3')
ax2.plot(epochs, [a * 100 for a in history['val_acc']],   label='Val',   color='#F44336')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_ylim(0, 105)

plt.suptitle('Training curves — MNIST 1D Conv1D', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact('training_curves.png')


In [ ]:
# Confusion matrix — logged as artifact to MLflow / MinIO
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        all_preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
        all_labels.extend(yb.numpy())

test_acc = accuracy_score(all_labels, all_preds)
print(f'Test accuracy: {test_acc:.3%}\n')
print(classification_report(all_labels, all_preds, digits=3))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=range(10), yticklabels=range(10),
    linewidths=0.5, linecolor='white', ax=ax,
)
ax.set_xlabel('Predicted digit', fontsize=12)
ax.set_ylabel('True digit', fontsize=12)
ax.set_title(f'Confusion Matrix — test accuracy {test_acc:.1%}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metric('test_accuracy', test_acc)
    for f in ['confusion_matrix.png', 'data_exploration.png', 'class_samples.png']:
        mlflow.log_artifact(f)
print(f'All artifacts logged to MLflow run {RUN_ID}')


In [ ]:
# t-SNE of learned feature representations
features, labels_tsne = [], []
model.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        features.append(model.encode(xb.to(DEVICE)).cpu().numpy())
        labels_tsne.extend(yb.numpy())

features = np.vstack(features)
labels_tsne = np.array(labels_tsne)

tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
emb = tsne.fit_transform(features)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(emb[:, 0], emb[:, 1],
                     c=labels_tsne, cmap='tab10', alpha=0.65, s=18)
plt.colorbar(scatter, ax=ax, label='Digit class')
ax.set_title('t-SNE of Conv1D learned features (test set)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
plt.tight_layout()
plt.savefig('tsne_features.png', dpi=120)
plt.show()

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact('tsne_features.png')
print('t-SNE artifact logged.')


## § 6 Deployment — Model Registry + DVC Dataset Versioning

In [ ]:
# Register model in MLflow Model Registry (stored in MinIO)
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))

with mlflow.start_run(run_id=RUN_ID):
    info = mlflow.pytorch.log_model(
        model,
        artifact_path='model',
        registered_model_name='mnist1d-conv1d',
    )

print(f'Model URI  : {info.model_uri}')
print(f'Registry   : {MLFLOW_TRACKING_URI}/#/models/mnist1d-conv1d')
print()
print('Load the model anywhere with:')
print(f"  model = mlflow.pytorch.load_model('{info.model_uri}')")


In [ ]:
# DVC — version the raw dataset and push to MinIO remote
# Run these commands on your host (not inside JupyterLab):
print("""
pip install 'dvc[s3]'

# In your project repo:
dvc init
dvc add mnist1d_data.pkl

# Configure the MinIO remote (values from ./scripts/start.sh dvc):
dvc remote add -d myremote s3://dvc-cache
dvc remote modify myremote endpointurl http://localhost:9010
dvc remote modify myremote access_key_id  minioadmin
dvc remote modify myremote secret_access_key minioadmin_secret

dvc push
git add mnist1d_data.pkl.dvc .gitignore
git commit -m 'Track MNIST 1D dataset with DVC'
""")


## Summary

| Step | Result |
|---|---|
| Dataset | MNIST 1D, 40-point sequences, 10 classes |
| Model | 3-layer Conv1D, ~30 K parameters |
| Training | 40 epochs, Adam + Cosine LR, ≥ 96 % val accuracy |
| Artifacts in MLflow/MinIO | confusion_matrix.png, training_curves.png, tsne_features.png, data_exploration.png, class_samples.png, model/ |
| Dataset versioned with | DVC → MinIO remote |

### Ideas for extending this stack

| Idea | Containers / tools |
|---|---|
| Hyperparameter optimisation | Optuna + MLflow autolog |
| Architecture comparison | MLflow experiment with MLP / LSTM / Conv1D runs |
| Distributed HPO | Ray Tune service (container) |
| REST inference API | FastAPI or BentoML loading the MLflow model URI |
| Model drift monitoring | Evidently AI + Grafana/Prometheus |
| Pipeline orchestration | Prefect or Dagster wiring DVC pull → train → push |
| Data annotation loop | Label Studio → export → DVC add → retrain |
| Feature store | Feast backed by Postgres/Redis |
| Vector search | Qdrant or Milvus for embedding similarity search |
| GPU serving | Triton Inference Server for production throughput |
